# 05. Data for the Power BI dashboard
Aggregated tables only: no client-level rows are published (Kaggle data rules).
Every table is rebuilt from the saved models and predictions of stages 3-4.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# sklearn (via src.scorecard) must be imported before lightgbm on this machine (vcomp140.dll)
from src.scorecard import gini, scorecard_table
from src.economics import client_profit, optimal_cutoff, portfolio_at_pd_cutoff, profit_curve
from src.modeling import load_bundle, to_lgbm_frame

PROCESSED = PROJECT_ROOT / "data" / "processed"
OUT = PROJECT_ROOT / "reports" / "powerbi"
OUT.mkdir(parents=True, exist_ok=True)
MARGIN, LGD = 0.10, 0.65
RANDOM_STATE = 42

MODELS = {
    "scorecard_with_ext": ("Scorecard (logreg on WoE)", "with EXT_SOURCE"),
    "scorecard_no_ext": ("Scorecard (logreg on WoE)", "no EXT_SOURCE"),
    "lightgbm_with_ext": ("LightGBM", "with EXT_SOURCE"),
    "lightgbm_no_ext": ("LightGBM", "no EXT_SOURCE"),
}

valid = pd.read_parquet(PROCESSED / "valid_predictions.parquet")
mart = pd.read_parquet(PROCESSED / "data_mart.parquet")
test = mart[mart["sample"] == "test"].reset_index(drop=True)
del mart

pd_test = {}
for key in MODELS:
    b = load_bundle(PROCESSED / "models" / f"{key}.joblib")
    X = b["woe"].transform(test)[b["features"]] if key.startswith("scorecard") else \
        to_lgbm_frame(test, b["features"], b["categories"])
    pd_test[key] = b["model"].predict_proba(X)[:, 1]
print(f"valid {len(valid):,}, test {len(test):,}")

valid 61,502, test 61,503


In [2]:
# 1. Profit curves on valid. ead_good / ead_bad let Power BI recompute profit for any margin and LGD:
#    profit = margin * ead_good - LGD * ead_bad
def score_from_pd(p):
    """Scorecard scale: 600 points at odds 50:1, +20 points double the odds."""
    factor = 20 / np.log(2)
    return 600 - factor * np.log(50) + factor * np.log((1 - p) / p)


curves = []
for key, (model_type, version) in MODELS.items():
    c = profit_curve(valid[f"pd_{key}"], valid["TARGET"], valid["AMT_CREDIT"], MARGIN, LGD, step=0.005)
    c.insert(0, "model", key)
    c.insert(1, "model_type", model_type)
    c.insert(2, "ext_source", version)
    c["approval_step"] = (c["approval_rate"] * 200).round() / 200  # exact 0.5% grid for Power BI slicers
    c["score_cutoff"] = score_from_pd(c["pd_cutoff"]).round(0) if key.startswith("scorecard") else np.nan
    curves.append(c)
profit_curves = pd.concat(curves, ignore_index=True)

# Baseline row per model is the last point (approval rate 100%); check against a direct sum
direct = client_profit(valid["TARGET"], valid["AMT_CREDIT"], MARGIN, LGD).sum()
assert np.allclose(profit_curves.groupby("model")["profit"].last(), direct)
profit_curves.to_csv(OUT / "profit_curve.csv", index=False)
profit_curves.groupby("model")[["profit"]].max().div(1e6).round(1)

,profit
model,
lightgbm_no_ext,1908.5
lightgbm_with_ext,2014.2
scorecard_no_ext,1809.5
scorecard_with_ext,1924.6


In [3]:
# 2. Sensitivity grid for all four models
rows = []
for lgd in [0.45, 0.55, 0.65, 0.75, 0.85]:
    for margin in [0.06, 0.08, 0.10, 0.12, 0.14]:
        base = client_profit(valid["TARGET"], valid["AMT_CREDIT"], margin, lgd).sum()
        for key in MODELS:
            best = optimal_cutoff(profit_curve(valid[f"pd_{key}"], valid["TARGET"], valid["AMT_CREDIT"], margin, lgd))
            rows.append({"model": key, "lgd": lgd, "margin": margin, "approve_all_profit": base,
                         "optimal_approval_rate": best["approval_rate"], "optimal_pd_cutoff": best["pd_cutoff"],
                         "optimal_profit": best["profit"], "uplift": best["profit"] - base})
sensitivity = pd.DataFrame(rows)
sensitivity.to_csv(OUT / "sensitivity.csv", index=False)
sensitivity.shape

(100, 8)

In [4]:
# 3. Model comparison: Gini with bootstrap CI on valid, final results on test at the valid cut-offs
rng = np.random.default_rng(RANDOM_STATE)
y_valid = valid["TARGET"].to_numpy()
boot = {key: [] for key in MODELS}
for _ in range(500):
    idx = rng.integers(0, len(y_valid), len(y_valid))
    for key in MODELS:
        boot[key].append(gini(y_valid[idx], valid[f"pd_{key}"].to_numpy()[idx]))

test_base = client_profit(test["TARGET"], test["AMT_CREDIT"], MARGIN, LGD).sum()
rows = []
for key, (model_type, version) in MODELS.items():
    valid_best = optimal_cutoff(profit_curves[profit_curves["model"] == key])
    on_test = portfolio_at_pd_cutoff(pd_test[key], test["TARGET"], test["AMT_CREDIT"],
                                     valid_best["pd_cutoff"], MARGIN, LGD)
    rows.append({
        "model": key, "model_type": model_type, "ext_source": version,
        "gini_valid": gini(valid["TARGET"], valid[f"pd_{key}"]),
        "gini_valid_ci_low": np.quantile(boot[key], 0.025), "gini_valid_ci_high": np.quantile(boot[key], 0.975),
        "gini_test": gini(test["TARGET"], pd_test[key]),
        "pd_cutoff": valid_best["pd_cutoff"],
        "approval_rate_valid": valid_best["approval_rate"], "profit_valid": valid_best["profit"],
        "approval_rate_test": on_test["approval_rate"], "bad_rate_approved_test": on_test["bad_rate"],
        "profit_test": on_test["profit"], "approve_all_profit_test": test_base,
        "uplift_test": on_test["profit"] - test_base,
    })
model_comparison = pd.DataFrame(rows)
model_comparison.to_csv(OUT / "model_comparison.csv", index=False)
model_comparison.round(4)

,model,model_type,ext_source,gini_valid,gini_valid_ci_low,gini_valid_ci_high,gini_test,pd_cutoff,approval_rate_valid,profit_valid,approval_rate_test,bad_rate_approved_test,profit_test,approve_all_profit_test,uplift_test
0,scorecard_with_ext,Scorecard (logreg on WoE),with EXT_SOURCE,0.5025,0.4887,0.5153,0.5089,0.1125,0.775,1.924591e+09,0.7741,0.0470,1.958617e+09,1.612812e+09,3.458054e+08
1,scorecard_no_ext,Scorecard (logreg on WoE),no EXT_SOURCE,0.4166,0.4008,0.4317,0.4231,0.1186,0.805,1.809521e+09,0.8066,0.0561,1.838995e+09,1.612812e+09,2.261838e+08
2,lightgbm_with_ext,LightGBM,with EXT_SOURCE,0.5532,0.5383,0.5661,0.5561,0.1357,0.835,2.014191e+09,0.8346,0.0490,2.029056e+09,1.612812e+09,4.162440e+08
3,lightgbm_no_ext,LightGBM,no EXT_SOURCE,0.4854,0.4710,0.4989,0.4929,0.1269,0.825,1.908500e+09,0.8253,0.0527,1.926715e+09,1.612812e+09,3.139031e+08


In [5]:
# 4. Fairness at each model's valid cut-off (valid sample; CODE_GENDER XNA excluded)
groups = {
    "gender": valid["CODE_GENDER"].where(valid["CODE_GENDER"].isin(["F", "M"])),
    "age": pd.cut(valid["age_years"], [0, 30, 40, 50, 60, 100], labels=["<30", "30-40", "40-50", "50-60", "60+"]),
}
rows = []
for key in ["scorecard_with_ext", "lightgbm_with_ext"]:
    cutoff = model_comparison.set_index("model").loc[key, "pd_cutoff"]
    for dim, grp in groups.items():
        d = valid.assign(group=grp, approved=valid[f"pd_{key}"] <= cutoff).dropna(subset=["group"])
        t = d.groupby("group", observed=True).agg(n=("TARGET", "size"), approval_rate=("approved", "mean"),
                                                  bad_rate=("TARGET", "mean"), mean_pd=(f"pd_{key}", "mean"))
        t["pd_to_bad_rate"] = t["mean_pd"] / t["bad_rate"]
        t["adverse_impact_ratio"] = t["approval_rate"] / t["approval_rate"].max()
        t = t.reset_index().assign(model=key, dimension=dim)
        rows.append(t)
fairness = pd.concat(rows, ignore_index=True)[["model", "dimension", "group", "n", "approval_rate", "bad_rate",
                                                "mean_pd", "pd_to_bad_rate", "adverse_impact_ratio"]]
fairness.to_csv(OUT / "fairness.csv", index=False)
fairness.round(3)

,model,dimension,group,n,approval_rate,bad_rate,mean_pd,pd_to_bad_rate,adverse_impact_ratio
0,scorecard_with_ext,gender,F,40450,0.802,0.070,0.075,1.071,1.000
1,scorecard_with_ext,gender,M,21051,0.724,0.101,0.091,0.903,0.903
2,scorecard_with_ext,age,<30,9089,0.554,0.114,0.127,1.113,0.601
3,scorecard_with_ext,age,30-40,16329,0.735,0.097,0.090,0.926,0.797
4,scorecard_with_ext,age,40-50,15388,0.808,0.076,0.074,0.971,0.876
5,scorecard_with_ext,age,50-60,13607,0.857,0.061,0.063,1.043,0.930
6,scorecard_with_ext,age,60+,7089,0.922,0.050,0.049,0.980,1.000
7,lightgbm_with_ext,gender,F,40450,0.861,0.070,0.073,1.042,1.000
8,lightgbm_with_ext,gender,M,21051,0.785,0.101,0.095,0.937,0.911
9,lightgbm_with_ext,age,<30,9089,0.710,0.114,0.114,1.002,0.749


In [6]:
# 5. Score bands of the main scorecard (valid): 10-point bands
s = valid["score_scorecard_with_ext"]
band_low = (np.floor(s / 10) * 10).clip(480, 640)  # thin tails pooled into the edge bands
bands = (valid.assign(band_low=band_low,
                      ead_bad=valid["AMT_CREDIT"] * valid["TARGET"])
         .groupby("band_low")
         .agg(n=("TARGET", "size"), n_bad=("TARGET", "sum"), mean_pd=("pd_scorecard_with_ext", "mean"),
              ead=("AMT_CREDIT", "sum"), ead_bad=("ead_bad", "sum"))
         .reset_index())
bands["band"] = bands["band_low"].astype(int).astype(str) + "-" + (bands["band_low"] + 9).astype(int).astype(str)
bands.loc[bands["band_low"] == 480, "band"] = "<490"
bands.loc[bands["band_low"] == 640, "band"] = "640+"
bands["bad_rate"] = bands["n_bad"] / bands["n"]
bands["ead_good"] = bands["ead"] - bands["ead_bad"]
bands["profit"] = MARGIN * bands["ead_good"] - LGD * bands["ead_bad"]
assert bands["n"].sum() == len(valid)
bands.to_csv(OUT / "score_bands.csv", index=False)
bands.round(3)

,band_low,n,n_bad,mean_pd,ead,ead_bad,band,bad_rate,ead_good,profit
0,480.0,159,67,0.532,6.431554e+07,29342259.0,<490,0.421,3.497328e+07,-1.557514e+07
1,490.0,342,139,0.423,1.462304e+08,60347434.5,490-499,0.406,8.588300e+07,-3.063753e+07
2,500.0,789,255,0.344,3.524347e+08,118064056.5,500-509,0.323,2.343707e+08,-5.330457e+07
3,510.0,1661,462,0.273,7.609502e+08,219317098.5,510-519,0.278,5.416331e+08,-8.839281e+07
4,520.0,2792,589,0.210,1.325975e+09,279585220.5,520-529,0.211,1.046389e+09,-7.709146e+07
5,530.0,4253,686,0.159,2.115382e+09,350561542.5,530-539,0.161,1.764821e+09,-5.138291e+07
6,540.0,5983,756,0.118,3.120916e+09,417038778.0,540-549,0.126,2.703877e+09,-6.875023e+05
7,550.0,7334,637,0.087,4.022652e+09,377663067.0,550-559,0.087,3.644989e+09,1.190179e+08
8,560.0,8203,523,0.063,4.754930e+09,336518239.5,560-569,0.064,4.418411e+09,2.231043e+08
9,570.0,8387,368,0.046,5.109708e+09,250937050.5,570-579,0.044,4.858771e+09,3.227680e+08


In [7]:
# 6. Scorecard points table (main model)
b = load_bundle(PROCESSED / "models" / "scorecard_with_ext.joblib")
card = scorecard_table(b["woe"].binnings_, b["model"], b["features"])
card["points"] = card["points"].round(0).astype(int)
card.to_csv(OUT / "scorecard.csv", index=False)
print(len(card), "rows;", card["feature"].nunique(), "features")
sorted(p.name for p in OUT.glob("*.csv"))

145 rows; 24 features


['fairness.csv',
 'model_comparison.csv',
 'profit_curve.csv',
 'score_bands.csv',
 'scorecard.csv',
 'sensitivity.csv']